In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
import numpy as np
import os

from astropy.time import Time

In [ ]:
# set the directory path to RACSLow (epoch 0)
directory_path = 'epoch_0/'

# create an empty list to store dataframes
dfs = []

# loop through all files in the directory that start with "beam_inf"
for filename in os.listdir(directory_path):
    if filename.startswith('beam_inf') and filename.endswith('.csv'):
        # read the CSV file into a pandas dataframe
        filepath = os.path.join(directory_path, filename)
        df = pd.read_csv(filepath)
        df['FIELD_NAME'] = filename.split('.')[0][-13:]
        
        # append the dataframe to the list
        dfs.append(df)
        
# concatenate all dataframes into a single dataframe
combined_df = pd.concat(dfs, ignore_index=True)


In [ ]:
# plot the beam time of each dataframe
plt.figure(figsize=(20, 5))
plt.scatter(combined_df['BEAM_TIME'], combined_df['BEAM_NUM'])
plt.title('Beam Time for RACS Low')
plt.xlabel('Time')
plt.ylabel('Beam Number')
plt.show()

In [ ]:
new_df = combined_df[combined_df['BEAM_TIME'].between(5.095e9, 5.098e9)]

plt.figure(figsize=(20, 5))
plt.scatter(new_df['BEAM_TIME'], new_df['BEAM_NUM'])
plt.title('Beam Time')
plt.xlabel('Time')
plt.ylabel('Beam Number')
plt.show()

In [ ]:
# create an empty list to store the max and min values
ra_dec_values = []

# loop through all dataframes in the list
for df in dfs:
    # find the max and min values of RA_DEG and DEC_DEG
    max_ra = df['RA_DEG'].max()
    min_ra = df['RA_DEG'].min()
    max_dec = df['DEC_DEG'].max()
    min_dec = df['DEC_DEG'].min()
    diff_ra = max_ra - min_ra
    diff_dec = max_dec - min_dec
    # append the max and min values to the list
    ra_dec_values.append({'max_ra': max_ra, 'min_ra': min_ra, 'max_dec': max_dec, 'min_dec': min_dec,
                           'diff_ra': diff_ra, 'diff_dec': diff_dec})

# create a new dataframe with the max and min values
ra_dec_df = pd.DataFrame(ra_dec_values)
print(ra_dec_df)


In [ ]:
# create an empty list to store the max and min values
beam_time_values = []

# loop through all dataframes in the list
for df in dfs:
    # find the max and min values of RA_DEG and DEC_DEG
    max_time = df['BEAM_TIME'].max()
    min_time = df['BEAM_TIME'].min()
    diff_time = max_time - min_time
    # append the max and min values to the list
    beam_time_values.append({'max_time': max_time, 'min_time': min_time, 'diff_time': diff_time})

# create a new dataframe with the max and min values
beam_time_df = pd.DataFrame(beam_time_values)
print(beam_time_df)

In [ ]:
df_field_data = pd.read_csv('epoch_0/field_data.csv')
print(df_field_data['SCAN_START'])

In [ ]:
# remove all the invalid entries from the dataframe
df_field_data = df_field_data[df_field_data['SCAN_START'] != -1]
df_field_data = df_field_data[df_field_data['COMMENT'].isnull()]
df_field_data = df_field_data[df_field_data['SCAN_LEN'] > 800]
df_field_data = df_field_data[df_field_data['DEC_DEG'] < 30]


# plot the scan start time against the scan time length for each scan
plt.figure(figsize=(20, 5))
plt.scatter(df_field_data['SCAN_START'], df_field_data['SCAN_LEN'])
plt.title('Scan Time')
plt.xlabel('Scan Start Time')
plt.ylabel('Scan Time Length')
plt.show()

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_field_data['RA_DEG'], df_field_data['DEC_DEG'], c=df_field_data['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()


In [ ]:
# plot the field vs the scan start time
plt.figure(figsize=(10, 8))
plt.subplot(111, projection="mollweide")
plt.scatter(np.radians(df_field_data['RA_DEG'])-np.pi, np.radians(df_field_data['DEC_DEG']), c=df_field_data['SCAN_START'], cmap='jet')
plt.title('Field vs Scan Start Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Start Time (in seconds)')
plt.show()

In [ ]:
# plot the field vs the scan start time
plt.figure(figsize=(10, 8))
plt.subplot(111, projection="mollweide")
plt.scatter(np.radians(df_field_data['GAL_LONG'])-np.pi, np.radians(df_field_data['GAL_LAT']), c=df_field_data['SCAN_START'], cmap='jet')
plt.title('Field vs Scan Start Time')
plt.xlabel('Galactic Longitude (in degrees)')
plt.ylabel('Galactic Latitude (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Start Time (in seconds)')
plt.grid(True)
plt.show()

In [ ]:
# create a new dataframe with fields around the galactic region and gaalctic cut separations of the catalogue
df_req_fields_gal = df_field_data[df_field_data['GAL_LAT'].between(-11, 11)]

In [ ]:
# create a new dataframe with fields where RACSlow1 correction is bad
df_req_fields1 = df_field_data[df_field_data['RA_DEG'].between(253, 262) & df_field_data['DEC_DEG'].between(4, 11)]
df_req_fields2 = df_field_data[df_field_data['RA_DEG'].between(330, 338) & df_field_data['DEC_DEG'].between(15, 23)]
df_req_fields3 = df_field_data[(df_field_data['RA_DEG'].between(355, 360) | df_field_data['RA_DEG'].between(0, 5)) & df_field_data['DEC_DEG'].between(15, 23)]
df_req_fields = pd.concat([df_req_fields1, df_req_fields2, df_req_fields3], ignore_index=True)

In [ ]:
df_req_fields = df_field_data[(df_field_data['DEC_DEG'] < -70) & ((df_field_data['GAL_LAT'] > 10) | (df_field_data['GAL_LAT'] < -10))]

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields['RA_DEG'], df_req_fields['DEC_DEG'], c=df_req_fields['SCAN_START'], cmap='jet')
plt.title('Field vs Scan Start Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-90, 90)
clb = plt.colorbar()
clb.set_label('Scan Start Time (in seconds)')
plt.show()

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by FIRST, and also have a scan start time of less than 5.065e9 seconds
df_req_fields = df_field_data[(df_field_data['SCAN_START'] < 5.065e9) & 
                       ((df_field_data['RA_DEG'].between(150, 225)) | (df_field_data['RA_DEG'] < 30) | (df_field_data['RA_DEG'] > 345)) & 
                       (df_field_data['DEC_DEG'].between(-10, 10)) &
                       (df_field_data['SCAN_LEN'].between(700, 1200))]


In [ ]:
# create a new dataframe with fields that already overlap with those mapped by FIRST
df_req_fields1 = df_field_data[((df_field_data['RA_DEG'].between(135, 240)) & (df_field_data['DEC_DEG'].between(-10, 30)))]
df_req_fields2 = df_field_data[(((df_field_data['RA_DEG'] < 45) | (df_field_data['RA_DEG'] > 315)) & (df_field_data['DEC_DEG'].between(-10, 10)))]
df_req_fields = pd.concat([df_req_fields1, df_req_fields2], ignore_index=True)

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by VLASS
df_req_fields = df_field_data[(df_field_data['DEC_DEG'].between(-35, 30)) & ((df_field_data['GAL_LAT'] > 10) | (df_field_data['GAL_LAT'] < -10))]
# df_req_fields = df_field_data[(df_field_data['DEC_DEG'].between(-35, 30))]
# df_req_fields = pd.concat([df_req_fields1, df_req_fields2], ignore_index=True)

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields['RA_DEG'], df_req_fields['DEC_DEG'], c=df_req_fields['SCAN_START'], cmap='jet')
plt.title('Field vs Scan Start Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-90, 90)
clb = plt.colorbar()
clb.set_label('Scan Start Time (in seconds)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields['RA_DEG'], df_req_fields['DEC_DEG'], c=df_req_fields['SCAN_START'], cmap='jet')
plt.title('Field vs Scan Start Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-90, 90)
clb = plt.colorbar()
clb.set_label('Scan Start Time (in seconds)')
plt.show()

In [ ]:
def mjd2utc(mjd_seconds):
    # create a Time object with the MJD seconds
    t = Time(mjd_seconds/86400, format='mjd', scale='utc')

    # convert the time to YYYYMMDD HH:MM:SS format
    time_str = t.datetime.strftime('%Y-%m-%d %H:%M:%S')
    fin_time = time_str + '.' + str(mjd_seconds).split('.')[1]
    return fin_time

# # define the MJD seconds
# mjd_seconds = df_req_fields.iloc[0]['SCAN_START']
# mjd = mjd_seconds / 86400

# # create a Time object with the MJD seconds
# t = Time(mjd, format='mjd', scale='utc')

# # convert the time to YYYYMMDD HH:MM:SS format
# time_str = t.datetime.strftime('%Y-%m-%d %H:%M:%S')
# fin_time = time_str + '.' + str(mjd_seconds).split('.')[1]
# print(fin_time)


In [ ]:
df_req_fields['UTC_SCAN_START'] = df_req_fields['SCAN_START'].apply(mjd2utc)

In [ ]:
# Convert 'UTC_SCAN_START' column to datetime type
df_req_fields['UTC_SCAN_START'] = pd.to_datetime(df_req_fields['UTC_SCAN_START'])

# Group the dataframe by date
# grouped_df = df_req_fields.groupby(df_req_fields['UTC_SCAN_START'].dt.date)

# # Iterate over each group
# for date, group in grouped_df:
#     # Print the date
#     print(date)
    
#     # Print the group
#     # print(group)
    
#     # Create bins for each group
#     bins = pd.cut(group['UTC_SCAN_START'], bins=10)
    
#     # Print the bins
#     print(bins)

date_counts = df_req_fields['UTC_SCAN_START'].dt.date.value_counts()

plt.figure(figsize=(10, 6))
plt.bar(date_counts.index, date_counts.values)
plt.title('Date vs Number of Occurrences')
plt.xlabel('Date')
plt.ylabel('Number of Occurrences')
plt.xticks(rotation=90)
plt.show()


In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_req_fields['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_req_fields['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_req_fields['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

time_list = df_req_fields['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

df_list = pd.DataFrame()

df_list['Field Name'] = field_list
df_list['SBID'] = sbid_list
df_list['CAL_SBID'] = cal_sbid_list
df_list['UTC Scan Start Time'] = time_list

df_list.sort_values(by='UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Save the field names to a numpy array for use in the crossmatch notebook
# Convert df_list to a numpy array
df_list_array = np.array(df_list)

# Save the numpy array to a file
np.save('RACSLow1_GalCut_DEC-90to-70.npy', df_list_array)


In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_req_fields_gal['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_req_fields_gal['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_req_fields_gal['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_req_fields_gal['UTC_SCAN_START'] = df_req_fields_gal['SCAN_START'].apply(mjd2utc)
time_list = df_req_fields_gal['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list_gal = pd.DataFrame()

# Assign values to the columns
df_list_gal['Field Name'] = field_list
df_list_gal['SBID'] = sbid_list
df_list_gal['CAL_SBID'] = cal_sbid_list
df_list_gal['UTC Scan Start Time'] = time_list

df_list_gal.sort_values('UTC Scan Start Time', inplace=True)
df_list_gal.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Save the field names to a numpy array for use in the crossmatch notebook
# Convert df_list to a numpy array
df_list_array = np.array(df_list_gal)

# Save the numpy array to a file
np.save('RACSLow1_GalList.npy', df_list_array)


In [ ]:
print(df_req_fields.iloc[0]['SCAN_START'])

In [ ]:
# Define the normal date time
date_time = fin_time

# Create a Time object with the normal date time
t = Time(date_time, format='iso', scale='utc')

# Convert the time to MJD in seconds
mjd_seconds = t.mjd * 86400

print(mjd_seconds)


# For full RACSLow1 coverage

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_field_data['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_field_data['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_field_data['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_field_data['UTC_SCAN_START'] = df_field_data['SCAN_START'].apply(mjd2utc)
time_list = df_field_data['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list = pd.DataFrame()

# Assign values to the columns
df_list['Field Name'] = field_list
df_list['SBID'] = sbid_list
df_list['CAL_SBID'] = cal_sbid_list
df_list['UTC Scan Start Time'] = time_list

df_list.sort_values('UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
%matplotlib inline

sbid = [str(df_list.iloc[i]['SBID']) for i in range(len(df_list))]

sbid_counts = Counter(sbid)

# for sbid, count in sbid_counts.items():
#     print(f"{sbid}: {count}")

sbids = list(sbid_counts.keys())
counts = list(sbid_counts.values())
consecutive_sbids = []
consecutive_counts = []

current_sbid = sbids[0]
current_count = counts[0]

for i in range(1, len(sbids)):
    if int(sbids[i]) == int(sbids[i-1]) + 1:
        current_count += counts[i]
    else:
        consecutive_sbids.append(str(current_sbid)+str('-')+str(sbids[i-1]))
        consecutive_counts.append(current_count)
        current_sbid = sbids[i]
        current_count = counts[i]

# Add the last consecutive sbid and count
consecutive_sbids.append(current_sbid)
consecutive_counts.append(current_count)

plt.figure(figsize=(20, 6))  # Set the figure size to 10 inches by 6 inches
plt.bar(consecutive_sbids, consecutive_counts)
plt.xlabel('SBIDs')
plt.ylabel('Number of Scans')
plt.title('SBID vs Number of Scans')
plt.xticks(rotation=90, ha='right')  # Rotate the labels by 45 degrees and align them to the right
# plt.xlim(left=0.4)
# plt.ylim(top=75)
plt.tight_layout()  # Adjust the layout to prevent overlapping labels
plt.show()

# plt.bar(sbids, counts)
# plt.xlabel('Date')
# plt.ylabel('Number of Scans')
# plt.title('Date vs Number of Scans')
# plt.xticks(rotation=90)
# plt.show()



In [ ]:
sbid = [str(df_list.iloc[i]['SBID']) for i in range(len(df_list))]
cal_sbid = [str(df_list.iloc[i]['CAL_SBID']) for i in range(len(df_list))]

cal_sbid_counts = Counter(cal_sbid)

cal_sbids = list(cal_sbid_counts.keys())
counts = list(cal_sbid_counts.values())

cal_sbid_to_sbid = {}

for i in range(len(df_list)):
    if cal_sbid[i] in cal_sbid_to_sbid:
        if sbid[i] not in cal_sbid_to_sbid[cal_sbid[i]]:
            cal_sbid_to_sbid[cal_sbid[i]].append(sbid[i])
    else:
        cal_sbid_to_sbid[cal_sbid[i]] = [sbid[i]]

# Print the corresponding SBID values for each CAL_SBID value
for cal_sbid_value, sbid_values in cal_sbid_to_sbid.items():
    print(f"CAL_SBID: {cal_sbid_value}, SBID values: {', '.join(sbid_values)}")

sbid_range = [f"{min(cal_sbid_to_sbid[cal_sbids[i]])}-{max(cal_sbid_to_sbid[cal_sbids[i]])}" for i in range(len(cal_sbids))]

plt.figure(figsize=(10, 6))
plt.bar(cal_sbids, counts)
plt.xlabel('SBID Range')
plt.ylabel('Number of Scans')
plt.title('SBID vs Number of Scans')
# Print the text at the bottom of the histogram
for i in range(len(sbid_range)):
    plt.text(i, counts[i], cal_sbids[i], ha='center', va='bottom', fontsize=8, rotation=90, mouseover=cal_sbids[i])

plt.xticks(range(len(sbid_range)), sbid_range, rotation=90)
plt.ylim(top=max(counts)+10)
plt.show()


In [ ]:
# Save the field names to a numpy array for use in the crossmatch notebook
# Convert df_list to a numpy array
df_list_array = np.array(df_list)

# Save the numpy array to a file
np.save('RACSLow1_FullList_v1.npy', df_list_array)


In [ ]:
date = [str(df_list.iloc[i]['UTC Scan Start Time'])[0:10] for i in range(len(df_list))]

date_counts = Counter(date)

for date, count in date_counts.items():
    print(f"{date}: {count}")

dates = list(date_counts.keys())
counts = list(date_counts.values())

plt.bar(dates, counts)
plt.xlabel('Date')
plt.ylabel('Number of Scans')
plt.title('Date vs Number of Scans')
plt.xticks(rotation=90)
plt.show()

